# Chapter 7 — Capabilities and Routing

**Book alignment:** current Chapter 7 · internal demo `Stage 06`

The shared demo package calls this **Stage 06** internally. The notebook number follows the book chapter number; the internal stage number is one lower.

**Question this notebook isolates:** Can the system expose and route only currently eligible capabilities while keeping selection separate from authorization?


## Hypothesis

Tool use improves when capabilities are explicit contracts and routing keeps **existence → eligibility → exposure → retrieval → selection → Stage-01 acceptance → execution → structured observation** inspectable. Exposure may reduce ambiguity, but it must never become authorization.

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "demo" / "agents-from-first-principles").exists():
            return candidate
    raise RuntimeError(
        "Run this notebook from a checkout containing demo/agents-from-first-principles"
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
DEMO_ROOT = REPO_ROOT / "demo" / "agents-from-first-principles"
TARGET_ROOT = DEMO_ROOT / "examples" / "broken-parser"
sys.path.insert(0, str(DEMO_ROOT))

from first_principles_agent.acceptance import AcceptanceStage, ActionAcceptanceBoundary
from first_principles_agent.actions import ActionKind
from first_principles_agent.capabilities import (
    Capability,
    CapabilityCase,
    CapabilityExecutor,
    CapabilityNeed,
    CapabilityOutcome,
    CapabilityPipeline,
    CapabilityRegistry,
    ExposurePolicy,
    evaluate_capability_routing,
)
from first_principles_agent.environment import RepositoryEnvironment
from first_principles_agent.runtime import RuntimeState

## Define a deliberately small action space

The contracts are data. The routing, exposure, retrieval, selection, acceptance and execution machinery lives in the shared implementation.

In [ ]:
registry = CapabilityRegistry(
    [
        Capability(
            "search_code",
            "find unknown file paths by query text",
            ActionKind.SEARCH_CODE,
            eligible_when=frozenset({"repo_available"}),
            keywords=frozenset(
                {
                    "search",
                    "code",
                    "unknown",
                    "path",
                    "query",
                    "text",
                    "logic",
                    "contains",
                }
            ),
            output_schema=("query", "matches"),
        ),
        Capability(
            "read_file",
            "read exact known path content",
            ActionKind.READ_FILE,
            eligible_when=frozenset({"repo_available", "path_known"}),
            keywords=frozenset({"read", "exact", "known", "path", "content", "file"}),
            output_schema=("path", "content"),
        ),
        Capability(
            "find_symbol",
            "resolve Python symbol definition",
            ActionKind.FIND_SYMBOL,
            eligible_when=frozenset({"repo_available"}),
            keywords=frozenset(
                {
                    "find",
                    "resolve",
                    "python",
                    "symbol",
                    "definition",
                    "function",
                    "class",
                }
            ),
            output_schema=("symbol", "locations"),
        ),
    ]
)
pipeline = CapabilityPipeline(registry, retrieval_k=2)

## Controlled routing experiment

The cases include three real tools plus explicit `ask_user`, `missing_precondition`, and `no_tool` outcomes.

In [ ]:
cases = [
    CapabilityCase(
        RuntimeState(facts=frozenset({"repo_available"})),
        CapabilityNeed("search code for unknown delimiter logic", target="delimiter"),
        CapabilityOutcome.TOOL,
        "search_code",
    ),
    CapabilityCase(
        RuntimeState(facts=frozenset({"repo_available", "path_known"})),
        CapabilityNeed("read known file content", target="parser.py"),
        CapabilityOutcome.TOOL,
        "read_file",
    ),
    CapabilityCase(
        RuntimeState(facts=frozenset({"repo_available"})),
        CapabilityNeed("find definition of Python symbol", target="split_record"),
        CapabilityOutcome.TOOL,
        "find_symbol",
    ),
    CapabilityCase(
        RuntimeState(),
        CapabilityNeed("which branch should I use?", requires_user_input=True),
        CapabilityOutcome.ASK_USER,
    ),
    CapabilityCase(
        RuntimeState(facts=frozenset({"repo_available"})),
        CapabilityNeed(
            "edit parser.py",
            target="parser.py",
            required_facts=frozenset({"write_permission"}),
        ),
        CapabilityOutcome.MISSING_PRECONDITION,
    ),
    CapabilityCase(
        RuntimeState(facts=frozenset({"repo_available"})),
        CapabilityNeed("summarize architecture diagram"),
        CapabilityOutcome.NO_TOOL,
    ),
]

metrics = evaluate_capability_routing(pipeline, cases)
metrics.as_dict()

In [ ]:
assert metrics.capability_coverage == 1.0
assert metrics.tool_recall_at_k == 1.0
assert metrics.selection_accuracy_given_available == 1.0
assert metrics.wrong_tool_rate == 0.0
assert metrics.abstention_accuracy == 1.0

unknown_path_state = RuntimeState(facts=frozenset({"repo_available"}))
known_path_state = RuntimeState(facts=frozenset({"repo_available", "path_known"}))
exposure = ExposurePolicy()
assert {c.name for c in exposure.expose(registry, unknown_path_state)} == {
    "search_code",
    "find_symbol",
}
assert {c.name for c in exposure.expose(registry, known_path_state)} == {
    "search_code",
    "read_file",
    "find_symbol",
}

## Exposure is not authorization

`read_file` can be exposed and selected while Stage 01 still rejects an unauthorized path. The environment must not run.

In [ ]:
unsafe = pipeline.route(
    known_path_state,
    CapabilityNeed("read known file content", target="../secret.txt"),
)
assert unsafe.selected and unsafe.selected.name == "read_file"


class MustNotExecute:
    def execute(self, action):
        raise AssertionError("authorization failure must prevent execution")


blocked = CapabilityExecutor().execute(
    unsafe,
    ActionAcceptanceBoundary(TARGET_ROOT),
    MustNotExecute(),
)
assert blocked.acceptance.stage == AcceptanceStage.AUTHORIZATION
assert blocked.observation is None
blocked.acceptance.reason

## Real repository observations

Now route two information needs through selection, Stage-01 acceptance, and the real broken-parser environment.

In [ ]:
executor = CapabilityExecutor()
boundary = ActionAcceptanceBoundary(TARGET_ROOT)
environment = RepositoryEnvironment(TARGET_ROOT)

search_decision = pipeline.route(
    unknown_path_state,
    CapabilityNeed("search code for unknown delimiter logic", target="delimiter"),
)
search_run = executor.execute(search_decision, boundary, environment)
assert search_run.observation is not None
assert any(
    item["path"] == "parser.py" for item in search_run.observation.data["matches"]
)

symbol_decision = pipeline.route(
    unknown_path_state,
    CapabilityNeed("find definition of Python symbol", target="split_record"),
)
symbol_run = executor.execute(symbol_decision, boundary, environment)
assert symbol_run.observation is not None
assert symbol_run.observation.data["locations"] == [
    {"path": "parser.py", "line": 1, "kind": "FunctionDef"}
]

useful_observation_rate = (
    sum([search_run.observation.ok, symbol_run.observation.ok]) / 2
)
assert useful_observation_rate == 1.0
{
    "search": search_run.observation.data,
    "symbol": symbol_run.observation.data,
    "useful_observation_rate": useful_observation_rate,
}

## The routing pipeline is not one decision

The current chapter names the separations explicitly:

```text
capability exists
→ capability is eligible in current state
→ capability is exposed
→ relevant capability is retrieved/discovered
→ one capability is selected
→ its concrete action crosses the action boundary
→ the environment executes and returns an observation
```

The experiments above exercise those boundaries separately. In particular, **selection does not imply authorization**, and a tool observation is evidence for later decisions rather than authority over the runtime.


## What was earned

The action space is now explicit and diagnosable. A capability can exist without being eligible, eligible without being exposed, exposed without being retrieved, retrieved without being selected, and selected without being authorized. Tool outputs are structured observations—evidence for later decisions, not authority.

Stage 07 will add a controlled lifecycle for selected past information.

## Demo API now implemented

```python
from first_principles_agent.capabilities import (
    CapabilityRegistry, ExposurePolicy, CapabilityRetriever, ToolSelector,
    CapabilityPipeline, CapabilityExecutor, ToolObservation,
)
decision = pipeline.route(state, need)
execution = executor.execute(decision, stage01_boundary, environment)
```